# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## llm 활용한 수정 기록 (제출용 실행본)

최신 `src/classification.py`는 분석 전에 데이터 계약을 검사하고, 위반이 있으면 `ValueError`로 멈춥니다. 공식 `data/processed` 데이터는 이 검사를 통과하지 못해 Notebook이 4번에서 중단되었습니다.

이 실행본에서는 **`src` 코드와 원래 `data/processed` 파일은 수정하지 않고**, 다음처럼 셀을 나누어 비교할 수 있게 했습니다.

| 위치 | 셀 | 내용 |
| --- | --- | --- |
| 4-1 | [원본 셀] | 공식 데이터로 그대로 실행하고, 실패하면 오류를 기록한 뒤 다음 셀로 넘어감 |
| 4-2 | [해결 셀] | 데이터 계약 위반 주문을 규칙별로 찾아 기록하고 분석 대상에서 제외 |
| 4-3 | [해결 셀] | 제외 후 데이터로 같은 공식 함수를 다시 실행 |
| 12-1 | [원본 셀] | 공식 `data/processed` 폴더로 전체 파이프라인 실행, 실패 기록 |
| 12-2 | [해결 셀] | 제외 후 데이터를 별도 폴더에 저장한 뒤 같은 파이프라인 실행 |
| 12-3 | [추가 셀] | 자동 Validation 결과와 공개 prediction 컬럼 확인 |

결측은 임의 값으로 채우지 않고(`fillna` 사용 안 함), 제외한 주문 수와 상태 분포를 규칙별로 남깁니다. 따라서 4-3 이후의 결과는 **공식 데이터 전체가 아니라 계약 위반 주문을 제외한 데이터** 기준입니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```

> ※ 수정 메모: `chapter10.md` STEP 0 기준으로 Chapter 10 입력은 프로젝트 루트에서 `python scripts/prepare_ch10_data.py`로 준비합니다. Notebook을 실행하기 직전에 이 스크립트를 먼저 실행하세요.


In [21]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\.venv\bin\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-course
전처리 데이터 폴더: C:\dev\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [22]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (752, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [23]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,180
1,cancelled,63
2,refunded,57


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


### 4-1. [원본 셀] 공식 데이터로 그대로 실행 (실패 기록)

공식 함수 호출은 원래 셀과 같습니다. 데이터 계약 검사에서 실패하면 오류 메시지를 `original_build_error`에 기록하고, Notebook 전체가 멈추지 않도록 다음 셀로 넘어갑니다.


In [24]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

# [원본 셀] 공식 데이터로 그대로 실행합니다.
# 데이터 계약 검사에서 실패하면 오류를 기록하고 다음 셀로 넘어갑니다.
try:
    (
        model_data,
        numeric_features,
        categorical_features,
        merge_checks,
        data_quality_checks,
    ) = build_classification_dataset(
        customers=customers,
        orders=orders,
        order_items=order_items,
    )
    original_build_error = None
    print('원본 데이터로 실행 성공')
except ValueError as error:
    original_build_error = str(error)
    print('원본 데이터로 실행 실패:', original_build_error)


원본 데이터로 실행 실패: orders 필수값에 결측치가 있습니다: 3건


### 4-2. [해결 셀] 데이터 계약 위반 주문 기록·제외

`chapter10.md` STEP 2~3의 계약을 순서대로 점검합니다. 위반한 주문은 값을 채우거나 고치지 않고 분석 대상에서 제외하며, 규칙별 제외 건수와 주문 상태 분포를 `exclusion_log`에 남깁니다.

| 순서 | 규칙 |
| --- | --- |
| 1 | 주문 필수값(주문 ID, 고객 ID, 주문일, 결제 수단, 주문 상태) 결측 |
| 2 | 주문 ID 중복 |
| 3 | 주문일 날짜 변환 실패 |
| 4 | 고객 ID 중복 또는 가입일 날짜 변환 실패 고객의 주문 |
| 5 | 고객 정보 미매칭 |
| 6 | 주문 상세 계산 불일치·결측 (`line_total = quantity × unit_price`) |
| 7 | 주문 상세가 없는 주문 |
| 8 | 가입일보다 주문일이 앞선 주문 (`days_since_signup < 0`) |

위반이 없는 규칙은 제외 건수가 0으로 기록됩니다.


In [25]:
# [해결 셀] 데이터 계약을 어기는 주문을 찾아 기록하고, 분석 대상에서 제외합니다.
# src/classification.py와 원래 data/processed 파일은 수정하지 않습니다.
import shutil

clean_customers = customers.copy()
clean_orders = orders.copy()
clean_items = order_items.copy()
exclusion_log = []


def exclude_orders(rule, mask):
    """mask가 True인 주문을 제외하고, 규칙별 제외 건수와 주문 상태 분포를 기록합니다."""
    global clean_orders
    removed = clean_orders.loc[mask]
    exclusion_log.append({
        'rule': rule,
        'excluded_orders': len(removed),
        'status_counts': removed['order_status'].value_counts(dropna=False).to_dict(),
    })
    clean_orders = clean_orders.loc[~mask].reset_index(drop=True)


# 1) 주문 필수값 결측
required_order_columns = ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
print('주문 필수값 결측(컬럼별):', clean_orders[required_order_columns].isna().sum().to_dict())
exclude_orders('1. 주문 필수값 결측', clean_orders[required_order_columns].isna().any(axis=1))

# 2) 주문 ID 중복
exclude_orders('2. 주문 ID 중복', clean_orders['order_id'].duplicated(keep=False))

# 3) 주문일 날짜 변환 실패
exclude_orders('3. 주문일 날짜 변환 실패', pd.to_datetime(clean_orders['order_date'], errors='coerce').isna())

# 4) 고객 ID 중복 또는 가입일 날짜 변환 실패 → 해당 고객과 그 고객의 주문 제외
bad_customer = (
    clean_customers['customer_id'].duplicated(keep=False)
    | pd.to_datetime(clean_customers['signup_date'], errors='coerce').isna()
)
bad_customer_ids = clean_customers.loc[bad_customer, 'customer_id']
clean_customers = clean_customers.loc[~bad_customer].reset_index(drop=True)
exclude_orders('4. 고객 정보 불량(ID 중복·가입일 변환 실패)', clean_orders['customer_id'].isin(bad_customer_ids))

# 5) 고객 정보 미매칭
exclude_orders('5. 고객 정보 미매칭', ~clean_orders['customer_id'].isin(clean_customers['customer_id']))

# 6) 주문 상세 계산 불일치·결측 → 해당 주문 전체 제외
quantity = pd.to_numeric(clean_items['quantity'], errors='coerce')
unit_price = pd.to_numeric(clean_items['unit_price'], errors='coerce')
line_total = pd.to_numeric(clean_items['line_total'], errors='coerce')
bad_items = (
    quantity.isna() | unit_price.isna() | line_total.isna()
    | ((line_total - quantity * unit_price).abs() > 1e-6)
)
print('계산 불일치·결측 주문 상세:', int(bad_items.sum()), '행')
exclude_orders('6. 주문 상세 계산 불일치·결측', clean_orders['order_id'].isin(clean_items.loc[bad_items, 'order_id']))

# 7) 주문 상세가 없는 주문
exclude_orders('7. 주문 상세 없음', ~clean_orders['order_id'].isin(clean_items['order_id']))

# 8) 가입일보다 주문일이 앞선 주문
signup_by_customer = pd.to_datetime(clean_customers.set_index('customer_id')['signup_date'])
days_since_signup = (
    pd.to_datetime(clean_orders['order_date']) - clean_orders['customer_id'].map(signup_by_customer)
).dt.days
exclude_orders('8. 가입일보다 앞선 주문', days_since_signup < 0)

# 남은 주문의 주문 상세만 유지
clean_items = clean_items[clean_items['order_id'].isin(clean_orders['order_id'])].reset_index(drop=True)

exclusion_log = pd.DataFrame(exclusion_log)
display(exclusion_log)
print('주문:', len(orders), '→', len(clean_orders), '건')
print('주문 상세:', len(order_items), '→', len(clean_items), '행')
print('고객:', len(customers), '→', len(clean_customers), '명')
display(
    clean_orders['order_status']
    .value_counts()
    .rename_axis('order_status')
    .reset_index(name='order_count')
)

# 12번 전체 파이프라인용: 제외 후 데이터를 별도 폴더에 저장합니다(원래 data/processed는 그대로 둠).
FILTERED_DIR = PROJECT_ROOT / 'data' / 'processed_ch10_contract_filtered'
FILTERED_DIR.mkdir(parents=True, exist_ok=True)
clean_customers.to_csv(FILTERED_DIR / 'customers_clean.csv', index=False)
clean_orders.to_csv(FILTERED_DIR / 'orders_clean.csv', index=False)
clean_items.to_csv(FILTERED_DIR / 'order_items_clean.csv', index=False)
if (PROCESSED_DIR / 'products_clean.csv').exists():
    shutil.copy(PROCESSED_DIR / 'products_clean.csv', FILTERED_DIR / 'products_clean.csv')
print('제외 후 데이터 저장 폴더:', FILTERED_DIR)


주문 필수값 결측(컬럼별): {'order_id': 0, 'customer_id': 0, 'order_date': 3, 'payment_method': 0, 'order_status': 0}
계산 불일치·결측 주문 상세: 0 행


,rule,excluded_orders,status_counts
0,1. 주문 필수값 결측,3,"{'completed': 2, 'refunded': 1}"
1,2. 주문 ID 중복,0,{}
2,3. 주문일 날짜 변환 실패,0,{}
3,4. 고객 정보 불량(ID 중복·가입일 변환 실패),6,{'completed': 6}
4,5. 고객 정보 미매칭,2,"{'completed': 1, 'cancelled': 1}"
5,6. 주문 상세 계산 불일치·결측,0,{}
6,7. 주문 상세 없음,1,{'completed': 1}
7,8. 가입일보다 앞선 주문,57,"{'completed': 32, 'cancelled': 14, 'refunded':..."


주문: 300 → 231 건
주문 상세: 752 → 573 행
고객: 150 → 148 명


,order_status,order_count
0,completed,138
1,cancelled,48
2,refunded,45


제외 후 데이터 저장 폴더: C:\dev\llm-data-analysis-course\data\processed_ch10_contract_filtered


### 4-3. [해결 셀] 제외 후 데이터로 같은 공식 함수 다시 실행

4-1과 같은 `build_classification_dataset()`를 그대로 사용하고, 입력만 4-2에서 정리한 데이터로 바꿉니다. 이후 5번부터의 셀은 이 결과를 사용합니다.


In [26]:
# [해결 셀] 계약 위반 주문을 제외한 데이터로 같은 공식 함수를 다시 실행합니다.
(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=clean_customers,
    orders=clean_orders,
    order_items=clean_items,
)

target_dist = target_distribution(model_data)

print('원본 데이터 실행 결과:', original_build_error or '성공')
print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


원본 데이터 실행 결과: orders 필수값에 결측치가 있습니다: 3건
모델링 데이터: (186, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,138,0.7419
1,1,cancelled,48,0.2581


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,186,186,True,0,PASS
1,customer_id → customers,186,186,True,0,PASS


,check_item,count
0,binary_target_rows,186
1,completed_rows,138
2,cancelled_rows,48
3,excluded_status_rows,45
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,138
9,status_scope:cancelled,48


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [27]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [28]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (111, 10)
X_valid: (37, 10)
X_test: (38, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,82,0.7387
1,train,1,cancelled,29,0.2613
2,validation,0,completed,28,0.7568
3,validation,1,cancelled,9,0.2432
4,test,0,completed,28,0.7368
5,test,1,cancelled,10,0.2632


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [29]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.486486,0.250000,0.555556,0.344828
1,Random Forest,validation,0.648649,0.166667,0.111111,0.133333
2,Dummy Most Frequent,validation,0.756757,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [30]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.3


,threshold,accuracy,precision,recall,f1
2,0.30,0.540541,0.346154,1.000000,0.514286
1,0.25,0.459459,0.310345,1.000000,0.473684
3,0.35,0.513514,0.320000,0.888889,0.470588
0,0.20,0.405405,0.290323,1.000000,0.450000
5,0.45,0.513514,0.285714,0.666667,0.400000
4,0.40,0.486486,0.272727,0.666667,0.387097
6,0.50,0.486486,0.250000,0.555556,0.344828
8,0.60,0.540541,0.214286,0.333333,0.260870
7,0.55,0.513514,0.200000,0.333333,0.250000
9,0.65,0.567568,0.181818,0.222222,0.200000


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [31]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.3,frozen_before_test,0.315789,0.233333,0.7,0.35


,pred_completed,pred_cancelled
actual_completed,5,23
actual_cancelled,3,7


,label,precision,recall,f1-score,support
0,completed,0.625000,0.178571,0.277778,28.000000
1,cancelled,0.233333,0.700000,0.350000,10.000000
2,accuracy,0.315789,0.315789,0.315789,0.315789
3,macro avg,0.429167,0.439286,0.313889,38.000000
4,weighted avg,0.521930,0.315789,0.296784,38.000000


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [32]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,18,0,1,0.334928,Logistic Regression,0.3
1,test_0002,41,0,0,0.279290,Logistic Regression,0.3
2,test_0003,3,0,1,0.498442,Logistic Regression,0.3
3,test_0004,184,0,1,0.435821,Logistic Regression,0.3
4,test_0005,172,0,1,0.541974,Logistic Regression,0.3


## 11. LLM 코드 검토 체크리스트


In [33]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


### 12-1. [원본 셀] 공식 `data/processed` 폴더로 실행 (실패 기록)


In [34]:
from src.classification import run_classification_analysis

# [원본 셀] 공식 data/processed 폴더로 전체 파이프라인을 실행합니다.
# 데이터 계약 검사에서 실패하면 오류를 기록하고 다음 셀로 넘어갑니다.
try:
    classification_result = run_classification_analysis(
        processed_dir=PROCESSED_DIR,
        report_dir=REPORT_DIR,
        random_state=42,
    )
    print('공식 processed 폴더로 실행 성공')
except ValueError as error:
    original_pipeline_error = str(error)
    print('공식 processed 폴더로 실행 실패:', original_pipeline_error)


공식 processed 폴더로 실행 실패: orders 필수값에 결측치가 있습니다: 3건


### 12-2. [해결 셀] 제외 후 데이터 폴더로 실행

4-2에서 저장한 `data/processed_ch10_contract_filtered` 폴더를 입력으로 같은 파이프라인을 실행합니다. 터미널의 `python scripts/run_classification_analysis.py`는 공식 `data/processed`를 읽으므로 12-1과 같은 이유로 실패합니다.


In [35]:
# [해결 셀] 계약 위반 주문을 제외한 데이터 폴더로 같은 파이프라인을 실행합니다.
classification_result = run_classification_analysis(
    processed_dir=FILTERED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.3


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.3,frozen_before_test,0.315789,0.233333,0.7,0.35


model_data_internal C:\dev\llm-data-analysis-course\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\dev\llm-data-analysis-course\reports\ch10_target_distribution.csv OK
feature_audit C:\dev\llm-data-analysis-course\reports\ch10_feature_audit.csv OK
merge_checks C:\dev\llm-data-analysis-course\reports\ch10_merge_checks.csv OK
data_quality_checks C:\dev\llm-data-analysis-course\reports\ch10_data_quality_checks.csv OK
split_summary C:\dev\llm-data-analysis-course\reports\ch10_split_summary.csv OK
validation_comparison C:\dev\llm-data-analysis-course\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\dev\llm-data-analysis-course\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\dev\llm-data-analysis-course\reports\ch10_test_metrics.csv OK
predictions_internal C:\dev\llm-data-analysis-course\reports\ch10_classification_predictions_internal.csv OK
predictions C:\dev\llm-data-analysis-course\reports\ch10_classification_predictions

### 12-3. [추가 셀] 자동 Validation 결과와 공개 prediction 확인


In [36]:
# 자동 Validation 결과 확인
validation_path = REPORT_DIR / 'ch10_classification_validation.csv'
if validation_path.exists():
    display(pd.read_csv(validation_path))
else:
    print('자동 Validation 파일이 없습니다:', validation_path)

# 공개 prediction에 식별자 컬럼이 없는지 확인
prediction_path = REPORT_DIR / 'ch10_classification_predictions.csv'
if prediction_path.exists():
    public_columns = pd.read_csv(prediction_path, nrows=0).columns.tolist()
    forbidden_columns = {'source_index', 'order_id', 'customer_id', 'product_id'}
    print('공개 prediction 컬럼:', public_columns)
    print('포함된 식별자 컬럼:', sorted(forbidden_columns & set(public_columns)) or '없음')
else:
    print('공개 prediction 파일이 없습니다:', prediction_path)


,check,value,status
0,target_exactly_completed_cancelled,"[np.int64(0), np.int64(1)]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS


공개 prediction 컬럼: ['record_id', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
포함된 식별자 컬럼: 없음


## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

## 제출 정보
- 이름: 배성윤
- GitHub ID: aeuyui
- 작성일: 2026-11-18
- 최종 노트북 URL: https://github.com/aeuyui/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

> **실행 상태 요약**
> - 공식 `data/processed` 데이터로 실행하면 최신 `src/classification.py`의 데이터 계약 검사에서 **"orders 필수값에 결측치가 있습니다: 3건"** 오류로 중단된다(노트북 4-1, 12-1).
> - `src` 코드와 원래 `data/processed`는 수정하지 않고, 노트북 4-2에서 데이터 계약 8가지를 점검해 위반 주문 69건을 규칙별로 기록·제외했다. 제외 후 데이터로 같은 공식 함수와 파이프라인을 다시 실행했고(4-3, 12-2), 자동 Validation 7개 항목이 모두 PASS였다(12-3).
> - 따라서 아래 결과는 **공식 데이터 전체가 아니라 계약 위반 주문을 제외한 데이터** 기준이다.

## 1. 분류 문제 정의
- 타깃: is_cancelled (주문 취소 여부)
- 0의 의미: completed (완료된 주문)
- 1의 의미: cancelled (취소된 주문)
- 제외한 상태: refunded (기타 상태는 데이터에 없음)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 접수된 직후, 주문이 완료 또는 취소로 결정되기 전
- 예측 시점에 사용할 수 있는 정보:
  - 주문 내용: item_count, total_quantity, order_amount
  - 주문 시기와 결제: order_month, order_dayofweek, payment_method
  - 고객 속성: gender, age, city, days_since_signup(가입 후 경과일)
- 예측 이후에만 알 수 있어 제외한 정보: order_status(주문 결과 자체), is_cancelled(타깃), 취소 사유/취소 시각 등 취소 이후 생기는 정보. 식별자(order_id, customer_id, product_id)와 고객명도 feature에서 제외했다.

### 나의 해석과 판단
주문이 들어온 직후 취소 가능성이 높은 주문을 미리 구분하는 상황을 가정해, 완료와 취소만 비교하는 이진 분류로 정했다. refunded는 0으로 바꾸지 않고 대상에서 제외했다. 환불은 주문이 완료된 뒤 되돌린 것이라 취소와 발생 시점과 의미가 다르고, 0(완료)에 섞으면 "완료된 주문"의 의미가 흐려지기 때문이다.

주문 내용(상품 수, 수량, 금액)은 주문이 접수될 때 이미 정해지는 정보이고, 타깃을 계산하는 데 사용되는 것도 아니므로 예측 시점에 쓸 수 있다고 판단했다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
age, item_count, total_quantity, order_amount, order_month, order_dayofweek, days_since_signup

### 사용한 범주형 Feature
gender, city, payment_method

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

노트북 5번의 누수 점검에서 사용 feature 10개에 order_status와 is_cancelled가 없었고("데이터 누수 컬럼 없음"), 식별자도 feature 목록에 없다. 자동 validation의 `forbidden_feature_overlap`도 0으로 PASS였다.

### 나의 해석과 판단
order_status는 주문 파일에 컬럼으로 존재하지만, 그 값은 주문이 완료되거나 취소된 뒤에 정해지는 결과이다. 이를 feature로 넣으면 모델이 예측이 아니라 정답을 읽게 된다. 컬럼이 파일에 있다는 것은 데이터를 모은 시점에 그 값이 있었다는 뜻일 뿐, 예측 시점에 알 수 있었다는 뜻은 아니다.

days_since_signup도 같은 문제를 보여 준다. 가입일은 고객 파일에 있는 값이지만, 주문일보다 가입일이 늦은 주문이 57건 있었다. 가입일이 나중에 기록되거나 바뀐 값이라면 주문 시점에는 그 가입일을 알 수 없었다는 뜻이므로, 이 주문들은 분석에서 제외했다(3번).

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0건 (노트북 4-2 "계산 불일치·결측 주문 상세 0행", 4-3 `line_total_mismatch` 0)
- 주문 특징 `order_id` 중복 건수: 0건 (4-2 규칙 2 "주문 ID 중복" 0건)

### 데이터 계약 위반 주문 기록 (노트북 4-2)

| 규칙 | 제외 주문 | 상태 분포 |
| --- | ---: | --- |
| 1. 주문 필수값 결측 (order_date 결측) | 3 | completed 2, refunded 1 |
| 2. 주문 ID 중복 | 0 | |
| 3. 주문일 날짜 변환 실패 | 0 | |
| 4. 고객 정보 불량 (고객 2명: ID 중복 또는 가입일 변환 실패) | 6 | completed 6 |
| 5. 고객 정보 미매칭 | 2 | completed 1, cancelled 1 |
| 6. 주문 상세 계산 불일치·결측 | 0 | |
| 7. 주문 상세 없음 | 1 | completed 1 |
| 8. 가입일보다 앞선 주문 | 57 | completed 32, cancelled 14, refunded 11 |
| 합계 | 69 | 주문 300 → 231건, 주문 상세 752 → 573행, 고객 150 → 148명 |

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 186 | 186 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 186 | 186 | 0 | PASS |

### 결과 관찰
- 공식 데이터는 주문일이 비어 있는 주문 3건 때문에 공식 함수에서 셀 실행이 중단되었다.
- 계약 위반 주문을 제외한 뒤에는 공식 함수가 정상 실행되었고, 품질 점검에서 `line_total_mismatch`, `negative_days_since_signup`, `missing_order_item_features`, `missing_customer_match`가 모두 0건이다. 두 병합 모두 행 수가 유지되고 미매칭이 0건이다.
- 이진 분류 대상(completed, cancelled)은 243건에서 186건으로 57건(약 23.5%) 줄었다. 가장 큰 원인은 가입일보다 앞선 주문(이진 대상 46건)이다.
- 제외 비율은 완료 주문 23.3%(42/180), 취소 주문 23.8%(15/63)로 비슷해, 취소 비율은 25.93%에서 25.81%로 거의 변하지 않았다.

### 나의 해석과 판단
미매칭은 "이 주문의 특징이나 고객 정보를 찾지 못했다"는 뜻이지, 값이 0이라는 뜻이 아니다. 원인 확인 없이 0으로 채우면 "상품 0개, 금액 0원인 주문"이나 "나이 0세 고객" 같은 존재하지 않는 데이터가 만들어지고, 모델이 이를 실제 패턴으로 학습한다. 또 키 오류나 데이터 누락 같은 원래 문제가 가려진다. 그래서 결측과 미매칭을 채우지 않고, 위반 주문을 규칙별 건수와 함께 기록한 뒤 제외했다.

다만 제외도 분석 결과를 바꾸는 결정이고, 이번 분석 같은 경우 이진 대상의 약 23.5%가 빠졌으므로 분석의 결과는 공식 데이터 전체를 대표하기 어렵다. 또 공식 함수는 첫 번째 오류에서 멈추기 때문에, 규칙 4, 5, 7로 제외한 주문을 공식 코드가 어떻게 처리하도록 의도했는지는 확인하지 못했다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 138 | 74.19% |
| 1 | cancelled | 48 | 25.81% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
- 분석 대상 186건 중 완료 주문이 138건(74.19%), 취소 주문이 48건(25.81%)으로, 완료가 취소의 약 2.9배이다.
- validation의 취소 주문은 9건, test의 취소 주문은 10건뿐이다.

### 나의 해석과 판단
모든 주문을 "완료"로 예측해도 accuracy가 약 74%이다. 따라서 accuracy가 74%보다 낮으면 아무것도 학습하지 않은 예측보다 정답률이 낮다는 뜻이고, 74%를 넘더라도 취소 주문을 실제로 찾았는지 알 수 없다. 이 분석의 목적은 취소 주문(1)을 찾는 것이므로, accuracy보다 취소 클래스의 precision, recall, F1을 중심으로 봐야 한다.

또 validation의 취소 주문이 9건뿐이라 1건을 더 맞히거나 틀리면 recall이 약 11.1%p, test에서는 10%p씩 바뀐다. 한 건마다 지표의 변화가 크기 때문에, 수치의 변화를 과대 해석하지 않게 주의해야 한다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.757 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.486 | 0.250 | 0.556 | 0.345 |
| Random Forest | 0.649 | 0.167 | 0.111 | 0.133 |

(validation 37건, 취소 9건, 기본 임계값 0.5. Precision, Recall, F1은 취소 클래스 기준)

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression (Dummy를 제외한 후보 중 validation F1이 가장 높음)

### 결과 관찰
- Dummy는 모든 주문을 완료로 예측해 accuracy가 0.757로 가장 높지만, 취소 주문을 하나도 찾지 못한다(recall 0).
- Logistic Regression은 취소 9건 중 5건을 찾았지만(recall 0.556), 취소로 예측한 20건 중 15건은 실제로 완료 주문이었다(precision 0.250). accuracy는 0.486으로 Dummy보다 크게 낮다.
- Random Forest는 취소 주문을 1건만 찾아 F1이 0.133으로 낮다.

### 나의 해석과 판단
취소 주문을 하나도 찾지 못하는 Dummy와 비교하면 Logistic Regression은 취소 주문을 일부 찾는다는 점에서만 낫다. 그러나 기본 임계값 0.5에서는 의미 있는 개선으로 보기 어렵다. 취소로 예측한 주문 중 실제 취소 비율(precision 0.250)이 validation의 취소 비율(0.243)과 거의 같아, 무작위로 고른 것과 차이가 없다. 모든 주문을 취소로 예측하는 단순 기준의 F1(약 0.391)보다도 F1(0.345)이 낮다.

### 한계와 추가 확인 사항
- validation이 37건(취소 9건)뿐이라 1건 차이로 recall이 약 11.1%p 바뀐다.
- Dummy는 "가장 많은 클래스 예측" 하나만 비교했다. "모두 취소로 예측"처럼 다른 단순 기준과도 함께 보아야 한다.
- 계약 위반 주문을 제외한 데이터 기준이라, 공식 데이터가 수정되면 결과가 달라질 수 있다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 82 | 29 |
| Validation | 모델·Threshold 선택 | 28 | 9 |
| Test | 선택 이후 최종 평가 | 28 | 10 |

세 split 모두 두 클래스가 존재하고(자동 Validation `all_splits_have_both_classes` PASS), 취소 비율이 24.3~26.3%로 비슷하다(stratified random split). 전처리(숫자형 중앙값 대체/표준화, 범주형 최빈값 대체/원-핫 인코딩)는 Pipeline 안에서 Train 데이터로만 학습한다.

### 나의 해석과 판단
모델과 threshold를 고르는 일은 주어진 데이터에 가장 잘 맞는 설정을 찾는 것이다. test를 보고 고르면 test 38건에 우연히 잘 맞는 설정을 고르게 되어, test 성능이 처음 보는 데이터에서의 성능을 대표하지 못하고 실제보다 좋게 보인다. 그래서 선택은 validation에서 끝내고, test는 모든 선택을 고정한 뒤 한 번만 사용했다(test 결과의 `selection_status`가 `frozen_before_test`).

### 교육용 random split의 한계
이번 분할은 날짜와 관계없이 클래스 비율만 맞춘 무작위 분할이라, 미래 주문이 train에 섞여 있을 수 있다. 실제 운영에서는 과거 주문으로 학습하고 이후 주문을 예측해야 하므로, 운영 전에 날짜 순서로 나눈 시간 순서 평가와, 학습 기간 이후의 새로운 기간에서 성능을 확인하는 out-of-time 검증을 추가해야 한다.

---

## 7. Validation 성능과 우선 지표

- Accuracy: 0.486
- Precision: 0.250
- Recall: 0.556
- F1: 0.345

(선택 모델 Logistic Regression, validation 37건, 기본 임계값 0.5)

### 내가 가장 중요하게 본 지표
F1을 가장 중요하게 보고, recall과 precision을 함께 확인했다. accuracy는 판단 기준에서 제외했다.

### 그 이유
- accuracy는 모든 주문을 완료로 예측해도 0.757이 나와 취소 주문을 찾는 능력을 보여 주지 못한다.
- 이 분석에는 취소를 놓치는 비용(FN)과 정상 주문을 취소로 잘못 분류하는 비용(FP)에 대한 정보가 없다. 따라서 어느 한쪽 오류만 줄이는 지표 대신, precision과 recall을 함께 반영하는 F1을 기준으로 삼았다.
- F1만 보면 precision과 recall 중 어느 쪽이 낮은지 가려지므로 두 지표를 함께 본다. 현재 모델은 recall(0.556)보다 precision(0.250)이 특히 낮아, 취소로 고른 주문 대부분이 정상 주문이다.

> 이 판단은 FP와 FN의 비용이 비슷하다는 가정에 기반한다. 예를 들어 취소를 놓치면 재고·배송 손실이 크다는 업무 정보가 있다면 recall을, 취소 위험 주문을 확인하는 비용이 크다면 precision을 더 중요하게 봐야 한다.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.25 | 0.310 | 1.000 | 0.474 | 29건을 취소로 예측해 취소 9건을 모두 찾지만, 정상 주문 20건도 함께 고름 |
| 0.30 | 0.346 | 1.000 | 0.514 | 26건을 취소로 예측해 취소 9건을 모두 찾고, 정상 주문 17건을 잘못 고름. F1이 가장 높음 |
| 0.50 | 0.250 | 0.556 | 0.345 | 기본값. 20건을 취소로 예측해 5건을 찾고 정상 주문 15건을 잘못 고름 |
| 0.60 | 0.214 | 0.333 | 0.261 | 14건을 취소로 예측해 3건만 맞히고 취소 6건을 놓침 |

(validation 37건, 취소 9건 기준)

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.30

### 선택 이유
7번에서 정한 우선 지표인 F1이 validation에서 0.514로 가장 높았다. 기본값 0.50과 비교하면 찾은 취소 주문은 5건에서 9건으로 늘고, 잘못 고른 정상 주문은 15건에서 17건으로 2건만 늘었다. 모든 주문을 취소로 예측하는 단순 기준의 F1(약 0.391)보다도 높다.

다만 0.30에서는 validation 37건 중 26건(약 70%)을 취소로 예측해 대부분의 주문을 취소 위험으로 표시한다. 또 취소 주문이 9건뿐이라 0.25(0.474), 0.35(0.471)와의 F1 차이는 1~2건의 차이로 바뀔 수 있는 수준이므로 불안정하다고 판단된다.

### 나의 해석과 판단
임계값도 모델을 고르는 선택이므로 validation에서 정해야 한다. final test를 보고 임계값을 고르면 test 데이터에 맞춰 조정한 것이 되어, test 성능이 실제보다 좋게 보이게 된다.

또 임계값을 낮추면 취소를 놓치지 않는 대신 확인할 주문이 크게 늘고(FP 증가), 높이면 확인할 주문은 줄지만 취소를 많이 놓친다(FN 증가). 실제 FP/FN 비용을 알면 F1이 아니라 그 비용에 맞춰 임계값을 validation에서 다시 정해야 한다.

---

## 9. FP/FN 해석

(final test 38건 혼동행렬: TN 5, FP 23, FN 3, TP 7)

- FP 의미: 실제로는 완료된 주문을 취소 위험으로 잘못 예측한 경우. Test에서 완료 주문 28건 중 23건(약 82.1%)이 해당한다.
- FN 의미: 실제로는 취소된 주문을 완료로 잘못 예측한 경우. Test에서 취소 주문 10건 중 3건(30.0%)을 놓쳤다.
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소 위험 주문을 미리 골라 확인하는 용도라고 가정하면 FN이 더 부담스러울 수 있다. 놓친 취소 주문은 재고 확보, 포장, 배송 준비 비용이 그대로 낭비될 수 있지만, FP는 정상 주문을 한 번 더 확인하는 비용에 그치기 때문이다. 반대로 모델 결과로 결제 보류나 배송 지연 같은 조치를 자동으로 한다면, 정상 고객에게 불편을 주는 FP가 더 부담스럽다.
- 그 판단에 추가로 필요한 업무 정보: 취소 1건이 발생했을 때의 비용(재고, 배송, 결제 수수료), 주문 1건을 확인하는 데 드는 비용, 취소가 주로 배송 전후 어느 시점에 일어나는지, 모델 결과를 어떤 조치에 쓸지

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
Test에서 모델은 38건 중 30건(약 79%)을 취소 위험으로 표시했고, 그중 23건(약 76.7%)이 정상 주문이었다. 취소 10건 중 7건을 찾았지만, 거의 모든 주문을 표시해서 얻은 결과라 걸러 낸다는 의미가 거의 없다. 확인 작업에 쓰면 정상 주문 대부분을 다시 확인하게 되고, 자동 조치에 쓰면 정상 고객 대부분에게 불편을 준다.

### 한계와 추가 확인 사항
- Test의 취소 주문이 10건뿐이라 FN 3건, TP 7건은 몇 건의 차이로 크게 바뀔 수 있다.
- 공개 예측 결과에는 실제값, 예측값, 취소 확률만 있어, FP와 FN이 어떤 주문에서 많이 생겼는지는 확인하지 못했다.
- 어느 오류가 더 부담스러운지는 가정에 따라 달라지며, 수업 데이터만으로 실제 업무 비용을 확정할 수 없다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.30
- Accuracy: 0.316
- Precision: 0.233
- Recall: 0.700
- F1: 0.350
- Validation 대비 변화: 같은 임계값(0.30)의 Validation 결과(accuracy 0.541, precision 0.346, recall 1.000, F1 0.514)와 비교하면 accuracy -0.225, precision -0.113, recall -0.300, F1 -0.164로 모두 크게 낮아졌다.

### 결과 관찰
- Test에서 모든 주문을 완료로 예측하면 accuracy가 0.737(28/38)로, 모델(0.316)보다 훨씬 높다.
- Test에서 모든 주문을 취소로 예측하면 F1이 약 0.417(precision 0.263, recall 1.0)로, 모델의 F1(0.350)보다 높다.
- 모델의 precision(0.233)은 Test의 취소 비율(0.263)보다도 낮다. 모델이 취소로 고른 주문이 무작위로 고른 주문보다 취소일 가능성이 낮았다.
- 공식 파이프라인을 제외 후 데이터로 다시 실행한 12-2에서도 같은 모델, 같은 임계값, 같은 Test 성능이 나왔다.

### 나의 해석과 판단
차이가 생긴 이유 후보는 다음과 같다.
1. Validation이 너무 작다. 취소 주문 9건으로 임계값을 골라, 그 9건을 모두 찾는 낮은 임계값(0.30)이 우연히 선택되었을 가능성이 크다.
2. 임계값이 낮아 대부분의 주문을 취소로 예측한다. 이런 설정은 데이터가 조금만 달라져도 precision이 크게 떨어진다.
3. 입력값의 설명력이 약하다. 기본 임계값에서도 Validation precision이 취소 비율과 거의 같아, 모델이 취소와 완료를 구분하는 신호를 거의 찾지 못했다.
4. 계약 위반 주문을 제외하면서 학습 데이터가 줄었다(Train 111건, 취소 29건).

어느 이유가 가장 큰지는 단정할 수 없다. 그러나 Test에서 모델이 "모두 취소로 예측"하는 단순 기준보다 F1이 낮고, precision이 취소 비율보다도 낮으므로, 이 모델이 처음 보는 주문에서 취소를 구분한다고 판단할 근거는 없다. 따라서 일반화 성능은 낮다고 판단했다. Test 결과가 기대보다 낮았지만, 같은 Test를 보고 모델이나 임계값을 다시 고르지 않았다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
- 공개 예측 파일(`ch10_classification_predictions.csv`)의 컬럼은 record_id, actual_is_cancelled, predicted_is_cancelled, cancel_probability, model, threshold이며, 식별자 컬럼이 없다(노트북 12-3, 자동 Validation `public_prediction_identifier_columns` PASS).
- source_index가 포함된 결과는 내부용 파일(`ch10_classification_predictions_internal.csv`)로 따로 저장되었다.
- 노트북 10번 셀의 `prediction_result`는 내부 오류 분석용 결과라 source_index가 함께 표시된다. 노트북을 공개 저장소에 올리므로, 이 셀의 출력도 공개된다는 점을 확인했다.

### 나의 해석과 판단
FP와 FN이 어떤 주문에서 생겼는지 분석하려면 원본 주문을 다시 찾아야 하므로, 내부 오류 분석에는 source_index 같은 연결 정보가 필요하다. 하지만 이 정보가 공개 결과에 남으면 원본 데이터의 다른 정보와 다시 연결해 특정 주문이나 고객을 추적할 수 있다. 그래서 오류 분석용 내부 자료와 공개 결과를 분리하고, 공개 결과에는 평가에 필요한 최소한의 컬럼만 남겨야 한다.

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

(12-2에서 모든 파일이 생성되었다. 위 체크는 내용이 노트북 출력으로 확인되는 파일이다.)

### 자동 Validation 결과

| check | value | status |
| --- | --- | --- |
| target_exactly_completed_cancelled | [0, 1] | PASS |
| forbidden_feature_overlap | 0 | PASS |
| merge_rows_and_matches | True | PASS |
| all_splits_have_both_classes | True | PASS |
| model_selected_on_validation | True | PASS |
| threshold_selected_on_validation | True | PASS |
| public_prediction_identifier_columns | none | PASS |

계약 위반 주문 69건을 제외한 데이터(`data/processed_ch10_contract_filtered`)로 실행한 결과, 7개 항목 모두 PASS이다.

노트북과 Script 비교: 공식 `data/processed`로 실행한 파이프라인(12-1, 터미널 스크립트와 같은 `run_classification_analysis()`)은 "orders 필수값에 결측치가 있습니다: 3건" 오류로 실패했다. 제외 후 데이터로 실행한 파이프라인(12-2)은 노트북 단계별 실행과 같은 모델(Logistic Regression), 같은 임계값(0.30), 같은 test 성능을 냈다.

### 자동 PASS가 의미하지 않는 것
자동 Validation PASS는 코드가 정한 규칙(타깃 범위, 금지 feature 제외, 병합 행 수, split 분리, 선택 순서, 공개 컬럼)을 지켰다는 뜻일 뿐이다. 다음은 자동 검증으로 판단할 수 없다.
- 운영 적합성: 이번 test에서 모델은 모두 취소로 예측하는 단순 기준보다 F1이 낮고, precision이 취소 비율보다도 낮았지만 7개 항목은 모두 PASS였다. 업무에 쓸 만한 성능인지는 사람이 판단해야 한다.
- 데이터 대표성: PASS는 계약 위반 주문을 제외한 뒤에 얻은 결과라, 제외한 23.5%가 다른 특성을 가졌다면 결과가 공식 데이터 전체를 대표하지 못한다.
- 시간에 따른 성능 변화: random split으로 평가했기 때문에 미래 주문에서도 같은 성능이 나오는지는 확인되지 않는다.
- 공정성: 성별, 나이, 도시 같은 고객 속성을 feature로 썼는데, 특정 고객 집단에 FP가 몰리는지는 검증하지 않았다.
- 예측과 원인의 구분: PASS는 모델이 취소 원인을 설명한다는 뜻이 아니다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. 클래스 비율과 Dummy baseline 대비 개선: 분석 대상의 74.19%가 완료 주문이라, 모두 완료로 예측하는 Dummy가 Test accuracy 0.737을 낸다. 모델의 Test accuracy는 0.316이다. 취소 클래스 기준으로는 Dummy(F1 0)보다 낫지만, 모두 취소로 예측하는 단순 기준(Test F1 약 0.417)보다 모델의 F1(0.350)이 낮다.
2. Precision/Recall/F1과 Final test 결과: Test에서 취소 10건 중 7건을 찾았지만(recall 0.700), 취소로 예측한 30건 중 23건이 정상 주문이었고 precision(0.233)이 취소 비율(0.263)보다 낮았다. Validation(F1 0.514)에 비해 Test(F1 0.350)에서 모든 지표가 크게 낮아져, Validation 성능이 처음 보는 데이터에서 유지되지 않았다.
3. Threshold 선택과 FP/FN 비용 가정: 임계값 0.30은 FP와 FN 비용이 비슷하다는 가정 아래 Validation F1이 가장 높아 선택했지만, 취소 9건으로 고른 값이고 대부분의 주문을 취소로 표시하는 설정이라 안정적인 선택이 아니다. 실제 비용 정보가 없어 업무에 맞는 임계값인지도 확인하지 못했다.
4. 데이터와 평가 방식의 한계: 공식 데이터는 데이터 계약을 통과하지 못했고, 이진 대상의 약 23.5%를 제외한 데이터로 분석했다. 교육용 stratified random split으로 평가해 미래 주문에서의 성능은 확인하지 못했다. 공개 예측 결과에서는 식별자가 제거되었다.

### 현재 모델의 가장 큰 위험
모델이 "취소 위험"이라고 표시하면 근거 있는 예측처럼 보이지만, Test에서는 거의 모든 주문을 취소 위험으로 표시했고, 표시한 주문이 실제로 취소일 가능성은 무작위로 고른 경우보다 낮았다. 이 결과를 믿고 결제 보류나 배송 지연 같은 조치를 하면 정상 고객 다수(Test에서 표시한 30건 중 23건)에게 불편을 주게 된다.

### 다음 개선 우선순위
1. 데이터 계약 문제 해결: 주문일 결측 3건, 고객 정보 불량 2명, 고객 미매칭, 주문 상세 없음, 가입일보다 앞선 주문 57건의 원인을 확인하고, 수정된 공식 데이터로 공식 스크립트가 그대로 통과하는지 확인한다.
2. 검증 방식 보완: 취소 주문이 9건뿐인 validation 대신 train 안의 교차검증으로 모델과 임계값의 안정성을 확인하고, "모두 취소로 예측" 기준을 baseline에 추가한다. 실제 운영 전에는 시간 순서 평가와 out-of-time 검증을 추가한다.
3. FP/FN 비용 정보 확보와 feature 보완: 취소 1건의 손실 비용과 주문 1건의 확인 비용을 업무 담당자에게 확인해 비용 기준으로 validation에서 임계값을 다시 정하고, 주문일 이전까지의 고객별 취소 이력처럼 예측 시점에 알 수 있으면서 취소와 관련 있을 만한 feature를 검토한다.

### 예측과 인과 구분
이번 모델에서 확인한 것은 "주문 내용, 주문 시기, 결제 수단, 고객 속성의 조합이 취소 주문과 함께 나타나는 정도"라는 예측 패턴뿐이다. 예를 들어 특정 결제 수단이나 금액대에서 취소 확률이 높게 나오더라도, 그 결제 수단이나 금액 때문에 고객이 주문을 취소한다고 말할 수 없다. 취소 사유, 배송 상황, 재고, 고객 문의 같은 정보가 데이터에 없고, 실험이나 인과 분석도 하지 않았기 때문이다. 또 현재 모델은 test에서 단순 기준보다 낫지 않아, 예측 패턴 자체도 확인되었다고 보기 어렵다.

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [ ] 최종 노트북 URL을 제출합니다.

(분석 결과와 자동 Validation PASS는 계약 위반 주문을 제외한 데이터 기준이다.)

---

## 부록. LLM 활용 및 결과 검증

- 사용한 LLM: Claude / 용도: 실행 오류 원인 분석, 데이터 계약 위반 주문을 기록·제외하는 노트북 셀 작성, 지표 해석과 답안 초안

| LLM 제안 | 실제 결과로 확인한 방법 | 판단 |
| --- | --- | --- |
| 오류가 날짜 형식 같은 코드 문제일 수 있으니 오류를 낸 코드를 확인할 것 | `src/classification.py`의 검사 코드를 직접 출력해, 날짜형 변환 후 음수 경과일을 막는 의도된 데이터 계약 검사임을 확인 | 채택 |
| `src` 검사를 끄지 말고, 노트북에서 계약 위반 주문을 규칙별로 기록·제외한 뒤 같은 공식 함수를 다시 실행 | 4-3에서 공식 함수가 통과하고 품질 점검(line_total 불일치, 음수 경과일, 미매칭)이 모두 0건, 12-3 자동 Validation 7개 항목 PASS | 채택 |
| 필수값 규칙에 결제 수단까지 포함 | 4-2의 컬럼별 결측이 order_date 3건뿐이라, 공식 함수가 보고한 3건과 같은 주문만 제외되었음을 확인 | 채택 |
| 실패 셀을 지우지 않고 try/except로 오류를 기록해 해결 셀과 나란히 둘 것 | 4-1, 12-1에 공식 데이터의 실패 메시지가 남아 해결 셀(4-3, 12-2)과 비교됨 | 채택 |
| 우선 지표를 F1로 하고 recall·precision을 함께 볼 것 | 노트북 8번의 선택 기준(F1 우선)과 일치하는지 확인 | 채택 |
| "모두 취소로 예측" 단순 기준과 비교할 것 | Test 혼동행렬의 클래스 수(완료 28, 취소 10)로 F1 약 0.417을 계산해 모델(0.350)과 비교 | 채택 |
| FN이 더 부담스러울 수 있으므로 임계값을 낮추는 방향 검토 | 비용 정보가 없어 확인 불가 | 보류. Test를 본 뒤 임계값을 바꾸지 않고, 비용 정보를 얻으면 Validation에서 다시 정하기로 함 |
